Importamos las librerias

In [ ]:
import os, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from mpl_toolkits.mplot3d import Axes3D

from matplotlib.animation import FuncAnimation
from sklearn.model_selection import train_test_split
from torchvision import datasets, transforms

Creamos directorios para guardar imgs y modelos

In [ ]:
os.makedirs('images', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('images/t-sne_thumbnails', exist_ok=True)
os.makedirs('gifs', exist_ok=True)

comprobamos si tenemos GPU / MPS

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
#device = 'mps' if torch.backends.mps.is_available() else 'cpu'
#print(device)

Descargamos el dataset de CelebA

In [ ]:
# Carpeta donde están tus imágenes
img_dir = "./data/celeba/img_align_celeba"

# Transformaciones
celeba_transform = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(32),
    transforms.Grayscale(num_output_channels=1),  
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Lista de imágenes
all_imgs = [f for f in os.listdir(img_dir) if f.endswith(".jpg")]
all_imgs.sort()  # opcional, para consistencia

# Split train/test (ej: 80% train, 20% test)
train_imgs, test_imgs = train_test_split(all_imgs, test_size=0.2, random_state=42)

# Función auxiliar para datasets a partir de lista de archivos
class CelebADataset(datasets.VisionDataset):
    def __init__(self, root, file_list, transform=None):
        super().__init__(root, transform=transform)
        self.root = root
        self.files = file_list

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        from PIL import Image
        img_path = os.path.join(self.root, self.files[idx])
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img

# Crear datasets
celeba_train = CelebADataset(img_dir, train_imgs, transform=celeba_transform)
celeba_test = CelebADataset(img_dir, test_imgs, transform=celeba_transform)

# Ejemplo de comprobación
print("Train size:", len(celeba_train))
print("Test size:", len(celeba_test))

In [ ]:
celeba_transform = transforms.Compose([
    # las imágenes originales son 218x178, si no recotamos la cara las estamos deformando
    transforms.CenterCrop(178),   # recorta la cara
    transforms.Resize(64),        # redimensionamos a 64x64
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5)),
])

celeba_train = torchvision.datasets.CelebA(
    root="./data",
    split="train",
    download=False,   
    transform=celeba_transform
)

celeba_test = torchvision.datasets.CelebA(
    root="./data",
    split="test",
    download=False,
    transform=celeba_transform
)

# Definimos el VAE

In [ ]:
# Encoder
class CelebAEncoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 4, 2, 1),   # 32 → 16
            nn.BatchNorm2d(32),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(32, 64, 4, 2, 1),  # 16 → 8
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(64, 128, 4, 2, 1), # 8 → 4
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, True)
        )

        self.fc_mu = nn.Linear(128*4*4, latent_dim)
        self.fc_logvar = nn.Linear(128*4*4, latent_dim)

    def forward(self, x):
        h = self.net(x)
        h = h.view(x.size(0), -1)
        return self.fc_mu(h), self.fc_logvar(h)

In [ ]:
# Decoder
class CelebADecoder(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()

        self.fc = nn.Linear(latent_dim, 128*4*4)

        self.net = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),  # 4 → 8
            nn.BatchNorm2d(64),
            nn.ReLU(True),

            nn.ConvTranspose2d(64, 32, 4, 2, 1),   # 8 → 16
            nn.BatchNorm2d(32),
            nn.ReLU(True),

            nn.ConvTranspose2d(32, 1, 4, 2, 1),    # 16 → 32
            nn.Tanh()
        )

    def forward(self, z):
        h = self.fc(z)
        h = h.view(z.size(0), 128, 4, 4)
        return self.net(h)

In [ ]:
# VAE
class CelebAVAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.enc = CelebAEncoder(latent_dim)
        self.dec = CelebADecoder(latent_dim)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.enc(x)
        z = self.reparameterize(mu, logvar)
        recon = self.dec(z)
        return recon, mu, logvar

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0, path='checkpoint.pth'):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.path = path

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        """Guarda el modelo si la pérdida mejora"""
        torch.save(model.state_dict(), self.path)

# Función pérdida del VAE

In [ ]:
def vae_loss(recon_x, x, mu, logvar, beta=1.0):
  recon = F.mse_loss(recon_x, x, reduction='sum')
  kl = torch.mean(-0.5 * torch.sum(1 + logvar - mu ** 2 - logvar.exp(), dim = 1))
  return recon + beta * kl, recon, kl

# Train Models

In [ ]:
def train_model(model, optimizer, train_set, epochs=100, plots=True, title=None):
    history_loss = []
    history_rec = []
    history_kl = []

    train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
    test_loader = DataLoader(celeba_test, batch_size=128, shuffle=False)
    n_batches = len(train_loader)

    early_stopping = EarlyStopping(patience=20, path=f'./models/{title}.pth')

    for ep in range(1, epochs + 1):
        model.train()
        tot_loss, tot_rec, tot_kl = 0.0, 0.0, 0.0

        for x in train_loader:
            x = x.to(device)
            recon, mu, logvar = model(x)
            loss, rec, kl = vae_loss(recon_x=recon, x=x, mu=mu, logvar=logvar)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            tot_loss += loss.item()
            tot_rec += rec.item()
            tot_kl += kl.item()

        avg_loss = tot_loss / n_batches
        avg_rec = tot_rec / n_batches
        avg_kl = tot_kl / n_batches

        history_loss.append(avg_loss)
        history_rec.append(avg_rec)
        history_kl.append(avg_kl)

        print(f"Epoch {ep:02d} | recon={avg_rec:.4f} kl={avg_kl:.4f}")

        
        model.eval()
        with torch.no_grad():
            x_test = next(iter(DataLoader(celeba_test, batch_size=16)))
            x_test = x_test.to(device)
            recon_test, _, _ = model(x_test)

        def denorm(t): return (t * 0.5 + 0.5).clamp(0, 1)
        x_img = denorm(x_test).cpu()
        recon_img = denorm(recon_test).cpu()

        fig, ax = plt.subplots(2, 16, figsize=(32, 4))
        for i in range(16):
            
            ax[0, i].imshow(x_img[i].squeeze(), cmap="gray")
            ax[0, i].axis("off")

            ax[1, i].imshow(recon_img[i].squeeze(), cmap="gray")
            ax[1, i].axis("off")

        fig.suptitle(f"Epoch {ep}")
        plt.tight_layout()
        plt.show()

        # Early Stopping
        val_loss_tot = 0
        with torch.no_grad():
            for x_v in test_loader:
                x_v = x_v.to(device)
                rv, mv, lv = model(x_v)
                v_loss, _, _ = vae_loss(rv, x_v, mv, lv)
                val_loss_tot += v_loss.item()

        avg_val_loss = val_loss_tot / len(test_loader)
        early_stopping(avg_val_loss, model)

        if early_stopping.early_stop:
            print(f"Early stopping en época {ep}. Recargando mejor modelo.")
            model.load_state_dict(torch.load(early_stopping.path))
            break

    
    if plots:
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        epocas_completadas = np.arange(1, len(history_loss) + 1)
        fig.suptitle(title)

        axes[0].plot(epocas_completadas, history_rec, label='Recon VAE', color='green')
        axes[0].set_title('Evolución de la Reconstrucción')
        axes[0].set_xlabel('Época')
        axes[0].set_ylabel('Error')
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        axes[1].plot(epocas_completadas, history_kl, label='Divergencia (KL)', color='red')
        axes[1].set_title('Evolución del Espacio Latente (KL)')
        axes[1].set_xlabel('Época')
        axes[1].set_ylabel('Valor KL')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend()

        plt.tight_layout()
        if title:
            plt.savefig(f'./images/{title}.png')
        plt.show()

    return history_rec, history_kl

### CelebA VAE con latent_dim = 128

In [ ]:
latent_dim = 128
model_latent_dim_128 = CelebAVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_128.parameters(), lr=1e-3)
epochs = 100

train_model(model_latent_dim_128, epochs=epochs, optimizer=opt, train_set=celeba_train, plots=True, title='Gráficas evolución VAE para CelebA con latent_dim = 128')

torch.save(model_latent_dim_128.state_dict(), './models/vae_celeba_ld_128.pth')

### CelebA VAE con latent_dim = 64

In [ ]:
latent_dim = 64
model_latent_dim_64 = CelebAVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_64.parameters(), lr=1e-3)
epochs = 100

train_model(model_latent_dim_64, epochs=epochs, optimizer=opt, train_set=celeba_train, plots=True, title='Gráficas evolución VAE para CelebA con latent_dim = 64')

torch.save(model_latent_dim_64.state_dict(), './models/vae_celeba_ld_64.pth')

### CelebA VAE con latent_dim = 32

In [ ]:
latent_dim = 32
model_latent_dim_32 = CelebAVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_32.parameters(), lr=1e-3)
epochs = 100

train_model(model_latent_dim_32, epochs=epochs, optimizer=opt, train_set=celeba_train, plots=True, title='Gráficas evolución VAE para CelebA con latent_dim = 32')

torch.save(model_latent_dim_32.state_dict(), './models/vae_celeba_ld_32.pth')

### CelebA VAE con latent_dim = 16

In [ ]:
latent_dim = 16
model_latent_dim_16 = CelebAVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_16.parameters(), lr=1e-3)
epochs = 100

train_model(model_latent_dim_16, epochs=epochs, optimizer=opt, train_set=celeba_train, plots=True, title='Gráficas evolución VAE para CelebA con latent_dim = 16')

torch.save(model_latent_dim_16.state_dict(), './models/vae_celeba_ld_16.pth')

### CelebA VAE con latent_dim = 8

In [ ]:
latent_dim = 8
model_latent_dim_8 = CelebAVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_8.parameters(), lr=1e-3)
epochs = 100

train_model(model_latent_dim_8, epochs=epochs, optimizer=opt, train_set=celeba_train, plots=True, title='Gráficas evolución VAE para CelebA con latent_dim = 8')

torch.save(model_latent_dim_8.state_dict(), './models/vae_celeba_ld_8.pth')

### CelebA VAE con latent_dim = 4

In [ ]:
latent_dim = 4
model_latent_dim_4 = CelebAVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_4.parameters(), lr=1e-3)
epochs = 100

train_model(model_latent_dim_4, epochs=epochs, optimizer=opt, train_set=celeba_train, plots=True, title='Gráficas evolución VAE para CelebA con latent_dim = 4')

torch.save(model_latent_dim_4.state_dict(), './models/vae_celeba_ld_4.pth')

### CelebA VAE con latent_dim = 2

In [ ]:
latent_dim = 2
model_latent_dim_2 = CelebAVAE(latent_dim).to(device)
opt = torch.optim.Adam(model_latent_dim_2.parameters(), lr=1e-3)
epochs = 100

train_model(model_latent_dim_2, epochs=epochs, optimizer=opt, train_set=celeba_train, plots=True, title='Gráficas evolución VAE para CelebA con latent_dim = 2')

torch.save(model_latent_dim_2.state_dict(), './models/vae_celeba_ld_2.pth')

# Carga pesos modelos

In [ ]:
latent_dim = 128
modelo_ld_128 = CelebAVAE(latent_dim).to(device)

modelo_ld_128.load_state_dict(torch.load("./models/vae_celeba_ld_128.pth", map_location=device))

modelo_ld_128.eval()

In [ ]:
latent_dim = 64
modelo_ld_64 = CelebAVAE(latent_dim).to(device)

modelo_ld_64.load_state_dict(torch.load("./models/vae_celeba_ld_64.pth", map_location=device))

modelo_ld_64.eval()

In [ ]:
latent_dim = 32
modelo_ld_32 = CelebAVAE(latent_dim).to(device)

modelo_ld_32.load_state_dict(torch.load("./models/vae_celeba_ld_32.pth", map_location=device))

modelo_ld_32.eval()

In [ ]:
latent_dim = 16
modelo_ld_16 = CelebAVAE(latent_dim).to(device)

modelo_ld_16.load_state_dict(torch.load("./models/vae_celeba_ld_16.pth", map_location=device))

modelo_ld_16.eval()

In [ ]:
latent_dim = 8
modelo_ld_8 = CelebAVAE(latent_dim).to(device)

modelo_ld_8.load_state_dict(torch.load("./models/vae_celeba_ld_8.pth", map_location=device))

modelo_ld_8.eval()

In [ ]:
latent_dim = 4
modelo_ld_4 = CelebAVAE(latent_dim).to(device)

modelo_ld_4.load_state_dict(torch.load("./models/vae_celeba_ld_4.pth", map_location=device))

modelo_ld_4.eval()

In [ ]:
latent_dim = 2
modelo_ld_2 = CelebAVAE(latent_dim).to(device)

modelo_ld_2.load_state_dict(torch.load("./models/vae_celeba_ld_2.pth", map_location=device))

modelo_ld_2.eval()

# Interpolation
- Para ello tomamos 2 imágenes del test set, las codificamos, y luego interpolamos linealmente en el espacio latente entre sus representaciones.
- Finalmente decodificamos cada punto de la interpolación para ver la transición entre ambas imágenes.

In [ ]:
def denorm(t): return(t* 0.5 + 0.5).clamp(0,1)

def interpolation(model):
  with torch.no_grad():
    model.eval()
    x = next(iter(DataLoader(celeba_test, batch_size=2)))
    x = x.to(device)
    recon, mu, logvar = model(x)
    z = model.reparameterize(mu, logvar) # Representaciones latentes de ambas imgs

    z1, z2 = z[0], z[1]
    n_interp = 10     # Nº de puntos de la interpolación (incluyendo extremos)
    interp_z = torch.stack([z1 * (1 - alpha) + z2 * alpha for alpha in torch.linspace(0, 1, n_interp)], dim=0)
    interp_x = model.dec(interp_z)
    interp_x = denorm(interp_x).cpu()
    fig, ax = plt.subplots(1, n_interp, figsize=(2*n_interp, 2))
    for i in range(n_interp):
        ax[i].imshow(interp_x[i].squeeze(), cmap="gray")
        ax[i].axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
interpolation(model=modelo_ld_128)

In [ ]:
interpolation(model=modelo_ld_64)

In [ ]:
interpolation(model=modelo_ld_32)

In [ ]:
interpolation(model=modelo_ld_16)

In [ ]:
interpolation(model=modelo_ld_8)

In [ ]:
interpolation(model=modelo_ld_4)

In [ ]:
interpolation(model=modelo_ld_2)

# Espacio latente t-SNE 2D

In [ ]:
def denorm(t): return (t * 0.5 + 0.5).clamp(0,1)

def latent_space(model, title_p1=None, title_p2=None):
    model.eval()

   
    with torch.no_grad():
        latents = []

        for x in DataLoader(celeba_test, batch_size=128):
            x = x.to(device)
            _, mu, _ = model(x)
            latents.append(mu.cpu())

        latents = torch.cat(latents, dim=0).numpy()

    tsne = TSNE(n_components=2, random_state=42)
    latents_2d = tsne.fit_transform(latents)

    plt.figure(figsize=(8, 8))
    plt.scatter(latents_2d[:, 0], latents_2d[:, 1], alpha=0.5)
    plt.title(title_p1)
    plt.grid(True, alpha=0.3)

    if title_p1:
        plt.savefig(f'./images/{title_p1}.png')

    plt.show()

    with torch.no_grad():
        n_samples = 1000
        latent_dim = model.enc.fc_mu.out_features

        z = torch.randn(n_samples, latent_dim).to(device)
        recon = model.dec(z)

        # volvemos a codificar
        _, mu, _ = model(recon)
        latents_prior = mu.cpu().numpy()

    tsne = TSNE(n_components=2, random_state=42)
    latents_2d = tsne.fit_transform(latents_prior)

    plt.figure(figsize=(8, 8))
    plt.scatter(latents_2d[:, 0], latents_2d[:, 1], alpha=0.5)
    plt.title(title_p2)
    plt.grid(True, alpha=0.3)

    if title_p2:
        plt.savefig(f'./images/{title_p2}.png')

    plt.show()

In [ ]:
latent_space(model=modelo_ld_128, title_p1='t-SNE of VAE CelebA Latent Space 128', title_p2='t-SNE of Prior Samples in Latent Space 128 Vae CelebA')

In [ ]:
latent_space(model=modelo_ld_64, title_p1='t-SNE of VAE CelebA Latent Space 64', title_p2='t-SNE of Prior Samples in Latent Space 64 Vae CelebA')

In [ ]:
latent_space(model=modelo_ld_32, title_p1='t-SNE of VAE CelebA Latent Space 32', title_p2='t-SNE of Prior Samples in Latent Space 32 Vae CelebA')

In [ ]:
latent_space(model=modelo_ld_16, title_p1='t-SNE of VAE CelebA Latent Space 16', title_p2='t-SNE of Prior Samples in Latent Space 16 Vae CelebA')

In [ ]:
latent_space(model=modelo_ld_8, title_p1='t-SNE of VAE CelebA Latent Space 8', title_p2='t-SNE of Prior Samples in Latent Space 8 Vae CelebA')

In [ ]:
latent_space(model=modelo_ld_4, title_p1='t-SNE of VAE CelebA Latent Space 4', title_p2='t-SNE of Prior Samples in Latent Space 4 Vae CelebA')

In [ ]:
latent_space(model=modelo_ld_2, title_p1='t-SNE of VAE CelebA Latent Space 2', title_p2='t-SNE of Prior Samples in Latent Space 2 Vae CelebA')

# t-SNE 3D

In [ ]:
def generar_y_descargar_gif(model, device, dataset, titulo):
    model.eval()
    latents = []

    with torch.no_grad():
        for x in DataLoader(dataset, batch_size=128):
            x = x.to(device)
            _, mu, _ = model(x)
            latents.append(mu.cpu())

    latents = torch.cat(latents, dim=0).numpy()

    # sample (para que no explote memoria)
    indices = np.random.choice(len(latents), 2000, replace=False)
    latents_sample = latents[indices]

    # t-SNE 3D
    tsne = TSNE(n_components=3, random_state=42)
    latents_3d = tsne.fit_transform(latents_sample)

    # Plot
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')

    def update(frame):
        ax.clear()
        ax.set_box_aspect([1,1,1])
        ax.set_axis_off()

        ax.scatter(
            latents_3d[:, 0],
            latents_3d[:, 1],
            latents_3d[:, 2],
            s=10,
            alpha=0.6
        )

        ax.set_title(titulo)
        ax.view_init(elev=20, azim=frame)

        return fig,

    frames = np.arange(0, 360, 2)
    ani = FuncAnimation(fig, update, frames=frames, interval=50)

    # Guardar GIF
    ani.save(f'./gifs/{titulo}.gif', writer='pillow', fps=20)
    plt.close()

In [ ]:
generar_y_descargar_gif(modelo_ld_128, device, celeba_test, titulo = 't_SNE_3D_ld_128')

In [ ]:
generar_y_descargar_gif(modelo_ld_64, device, celeba_test, titulo = 't_SNE_3D_ld_64')

In [ ]:
generar_y_descargar_gif(modelo_ld_32, device, celeba_test, titulo = 't_SNE_3D_ld_32')

In [ ]:
generar_y_descargar_gif(modelo_ld_16, device, celeba_test, titulo = 't_SNE_3D_ld_6')

In [ ]:
generar_y_descargar_gif(modelo_ld_8, device, celeba_test, titulo = 't_SNE_3D_ld_8')

In [ ]:
generar_y_descargar_gif(modelo_ld_4, device, celeba_test, titulo = 't_SNE_3D_ld_4')

In [ ]:
generar_y_descargar_gif(modelo_ld_2, device, celeba_test, titulo = 't_SNE_3D_ld_2')

El fallo pasa porque t-SNE no puede crear más dimensiones de las que ya tiene la entrada. El vector latente tiene 2 dimensiones, pero se esrá intentando proyectarlo a 3D. Como solo hay 2 valores por punto, no se puede generar una tercera dimensión.

## t-SNE con reconstrucciones

In [ ]:
@torch.no_grad()
def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1)


def pick_spread_points(emb, n_show=150, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(emb.shape[0])
    rng.shuffle(idx)

    chosen = []
    thr = 0.02 * np.var(emb, axis=0).sum()

    for i in idx:
        if not chosen:
            chosen.append(i)
            continue

        d2 = np.min(np.sum((emb[chosen] - emb[i])**2, axis=1))
        if d2 > thr:
            chosen.append(i)

        if len(chosen) >= n_show:
            break

    return np.array(chosen)


@torch.no_grad()
def tsne_with_recon_thumbnails(model, loader, device,
                              n_total=3000, n_show=150,
                              zoom=0.8, seed=42, title=None):

    import os
    os.makedirs('./images/t-sne_thumbnails', exist_ok=True)

    model.eval()

    MU = []
    RECON = []

    for x in loader:   
        x = x.to(device)
        recon, mu, _ = model(x)

        MU.append(mu.cpu())
        RECON.append(recon.cpu())

        if sum(t.size(0) for t in MU) >= n_total:
            break

    MU = torch.cat(MU)[:n_total].numpy()
    RECON = torch.cat(RECON)[:n_total]

    # t-SNE
    emb = TSNE(n_components=2, random_state=seed,
               init="pca", learning_rate="auto").fit_transform(MU)

    chosen = pick_spread_points(emb, n_show=n_show, seed=seed)

   
    thumbs = denorm(RECON[chosen])[:, 0].numpy()

    # Plot
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

    for k, i in enumerate(chosen):
        ab = AnnotationBbox(
            OffsetImage(thumbs[k], zoom=zoom, cmap="gray"),
            (emb[i, 0], emb[i, 1]),
            frameon=False
        )
        ax.add_artist(ab)

    ax.set_xlim(emb[chosen, 0].min() - 5, emb[chosen, 0].max() + 5)
    ax.set_ylim(emb[chosen, 1].min() - 5, emb[chosen, 1].max() + 5)

    plt.tight_layout()

    if title:
        plt.savefig(f'./images/t-sne_thumbnails/{title}.png')

    plt.show()

    
testloader = DataLoader(celeba_test, batch_size=128, shuffle=False)

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_128, testloader, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST LS 128 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_64, testloader, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST LS 64 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_32, testloader, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST LS 32 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_16, testloader, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST LS 16 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_8, testloader, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST LS 8 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_4, testloader, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST LS 4 Img')

In [ ]:
tsne_with_recon_thumbnails(modelo_ld_2, testloader, device, n_total=3000, n_show=150, zoom=0.9, title='t-SNE of VAE MNIST LS 2 Img')